# 📘 Notebook 8: Magic Methods (Dunder Methods) Deep Dive

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. Understand what **dunder methods** are and how Python uses them
2. Implement **string representation** (`__repr__`, `__str__`, `__format__`)
3. Implement **comparison** operators (`__eq__`, `__lt__`, `__gt__`, etc.)
4. Implement **arithmetic** operators (`__add__`, `__sub__`, `__mul__`, etc.)
5. Make objects **callable**, **iterable**, and support **context managers**
6. Implement **`__getitem__`**, **`__contains__`**, and **`__len__`**

---

## 1. What Are Dunder Methods?

**Dunder methods** (double underscore methods, also called **magic methods** or **special methods**) are methods with names surrounded by double underscores: `__name__`.

They let you define how your objects interact with **Python's built-in operations**:

```
When you write:          Python calls:
─────────────────        ──────────────
str(obj)            →    obj.__str__()
len(obj)            →    obj.__len__()
obj1 + obj2         →    obj1.__add__(obj2)
obj1 == obj2        →    obj1.__eq__(obj2)
obj[key]            →    obj.__getitem__(key)
obj()               →    obj.__call__()
for x in obj:       →    obj.__iter__()
```

> 💡 You **never** call dunder methods directly (like `obj.__add__(other)`). Instead, use the corresponding operator or function.

---

## 2. String Representation: `__repr__` vs `__str__`

| Method | Purpose | Called By | Audience |
|--------|---------|-----------|----------|
| `__repr__` | Unambiguous, for debugging | `repr()`, interactive console | **Developers** |
| `__str__` | Readable, for display | `str()`, `print()` | **Users** |
| `__format__` | Custom formatting | `format()`, f-strings | **Users** |

> 💡 If only one is defined, implement `__repr__`. Python falls back to `__repr__` when `__str__` is missing.

In [ ]:
class Product:
    def __init__(self, name: str, price: float, sku: str):
        self.name = name
        self.price = price
        self.sku = sku
    
    def __repr__(self):
        """For developers: unambiguous, shows how to recreate the object."""
        return f"Product('{self.name}', {self.price}, '{self.sku}')"
    
    def __str__(self):
        """For users: readable, friendly display."""
        return f"{self.name} — ${self.price:.2f}"
    
    def __format__(self, format_spec):
        """Custom formatting options."""
        if format_spec == "short":
            return f"{self.name} (${self.price})"
        elif format_spec == "full":
            return f"{self.name} [SKU: {self.sku}] — ${self.price:.2f}"
        else:
            return str(self)


p = Product("Wireless Mouse", 29.99, "WM-001")

print(f"repr(): {repr(p)}")           # __repr__
print(f"str():  {str(p)}")            # __str__
print(f"print(): {p}")                # __str__ (print uses str())
print(f"short:  {p:short}")           # __format__ with "short"
print(f"full:   {p:full}")            # __format__ with "full"

# In a list, repr() is used for each element
products = [Product("Mouse", 29.99, "M1"), Product("Keyboard", 59.99, "K1")]
print(f"\nList: {products}")          # Uses __repr__ for each item

---

## 3. Comparison Methods

| Method | Operator | Example |
|--------|----------|---------|
| `__eq__` | `==` | `a == b` |
| `__ne__` | `!=` | `a != b` |
| `__lt__` | `<` | `a < b` |
| `__le__` | `<=` | `a <= b` |
| `__gt__` | `>` | `a > b` |
| `__ge__` | `>=` | `a >= b` |

> 💡 **Pro Tip**: Implement `__eq__` and `__lt__`, then use `@functools.total_ordering` to auto-generate the rest!

In [ ]:
from functools import total_ordering

@total_ordering  # Auto-generates __le__, __gt__, __ge__ from __eq__ and __lt__
class Student:
    def __init__(self, name: str, gpa: float):
        self.name = name
        self.gpa = gpa
    
    def __eq__(self, other):
        if not isinstance(other, Student):
            return NotImplemented  # Let Python handle it
        return self.gpa == other.gpa
    
    def __lt__(self, other):
        if not isinstance(other, Student):
            return NotImplemented
        return self.gpa < other.gpa
    
    def __repr__(self):
        return f"Student('{self.name}', gpa={self.gpa})"


alice = Student("Alice", 3.9)
bob = Student("Bob", 3.5)
charlie = Student("Charlie", 3.9)

print(f"{alice.name} == {charlie.name}: {alice == charlie}")   # Same GPA
print(f"{alice.name} > {bob.name}:  {alice > bob}")            # auto-generated!
print(f"{bob.name} <= {alice.name}: {bob <= alice}")           # auto-generated!

# Now we can sort!
students = [alice, bob, charlie, Student("Diana", 4.0)]
print(f"\nSorted by GPA: {sorted(students)}")
print(f"Highest GPA: {max(students)}")

---

## 4. Arithmetic Methods

In [ ]:
class Matrix:
    """A simple 2x2 matrix with operator support."""
    
    def __init__(self, data: list):
        """data should be [[a, b], [c, d]]"""
        assert len(data) == 2 and len(data[0]) == 2, "Must be 2x2"
        self.data = data
    
    def __add__(self, other):
        """Matrix addition: M1 + M2"""
        return Matrix([
            [self.data[i][j] + other.data[i][j] for j in range(2)]
            for i in range(2)
        ])
    
    def __sub__(self, other):
        """Matrix subtraction: M1 - M2"""
        return Matrix([
            [self.data[i][j] - other.data[i][j] for j in range(2)]
            for i in range(2)
        ])
    
    def __mul__(self, scalar):
        """Scalar multiplication: M * 3"""
        return Matrix([
            [self.data[i][j] * scalar for j in range(2)]
            for i in range(2)
        ])
    
    def __rmul__(self, scalar):
        """Reverse multiplication: 3 * M (called when left operand doesn't support it)"""
        return self.__mul__(scalar)
    
    def __neg__(self):
        """Unary negation: -M"""
        return self * -1
    
    def __repr__(self):
        return f"Matrix({self.data})"
    
    def __str__(self):
        row1 = f"| {self.data[0][0]:>4} {self.data[0][1]:>4} |"
        row2 = f"| {self.data[1][0]:>4} {self.data[1][1]:>4} |"
        return f"{row1}\n{row2}"


m1 = Matrix([[1, 2], [3, 4]])
m2 = Matrix([[5, 6], [7, 8]])

print("M1:")
print(m1)
print("\nM2:")
print(m2)
print("\nM1 + M2:")
print(m1 + m2)
print("\nM1 * 3:")
print(m1 * 3)
print("\n2 * M2 (reverse mul):")
print(2 * m2)
print("\n-M1:")
print(-m1)

---

## 5. Container Methods: `__len__`, `__getitem__`, `__contains__`

Make your objects behave like containers (lists, dicts):

In [ ]:
class Playlist:
    """A playlist that behaves like a container."""
    
    def __init__(self, name: str):
        self.name = name
        self._songs = []
    
    def add(self, song: str):
        self._songs.append(song)
        return self  # Enable chaining
    
    def __len__(self):
        """len(playlist)"""
        return len(self._songs)
    
    def __getitem__(self, index):
        """playlist[0], playlist[1:3], supports slicing!"""
        return self._songs[index]
    
    def __setitem__(self, index, value):
        """playlist[0] = 'New Song'"""
        self._songs[index] = value
    
    def __delitem__(self, index):
        """del playlist[0]"""
        del self._songs[index]
    
    def __contains__(self, song):
        """'Song' in playlist"""
        return song.lower() in [s.lower() for s in self._songs]
    
    def __iter__(self):
        """for song in playlist:"""
        return iter(self._songs)
    
    def __repr__(self):
        return f"Playlist('{self.name}', {len(self)} songs)"


# Build a playlist
playlist = Playlist("Road Trip")
playlist.add("Bohemian Rhapsody").add("Hotel California").add("Stairway to Heaven")
playlist.add("Sweet Child O' Mine").add("Comfortably Numb")

# Use like a container!
print(f"Playlist: {playlist}")
print(f"Length: {len(playlist)}")
print(f"First: {playlist[0]}")
print(f"Last: {playlist[-1]}")
print(f"Slice: {playlist[1:3]}")
print(f"Contains 'Hotel California'? {'Hotel California' in playlist}")
print(f"Contains 'Thriller'? {'Thriller' in playlist}")

print("\n🎵 Now playing:")
for i, song in enumerate(playlist, 1):
    print(f"  {i}. {song}")

---

## 6. Making Objects Callable: `__call__`

`__call__` lets you use an object as if it were a function:

In [ ]:
class Multiplier:
    """An object that multiplies numbers by a fixed factor."""
    
    def __init__(self, factor: float):
        self.factor = factor
    
    def __call__(self, value: float) -> float:
        """Makes the object callable: multiplier(5)"""
        return value * self.factor
    
    def __repr__(self):
        return f"Multiplier({self.factor})"


double = Multiplier(2)
triple = Multiplier(3)
half = Multiplier(0.5)

# Use objects as functions!
print(f"double(5) = {double(5)}")
print(f"triple(5) = {triple(5)}")
print(f"half(10) = {half(10)}")

# Works with map, filter, etc.
numbers = [1, 2, 3, 4, 5]
print(f"\nDoubled: {list(map(double, numbers))}")
print(f"Tripled: {list(map(triple, numbers))}")

In [ ]:
# A more practical example: a counter/tracker
class CallCounter:
    """Wraps a function and counts how many times it's called."""
    
    def __init__(self, func):
        self.func = func
        self.count = 0
    
    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.func(*args, **kwargs)
    
    def __repr__(self):
        return f"CallCounter({self.func.__name__}, calls={self.count})"


@CallCounter  # This is actually a decorator!
def greet(name):
    return f"Hello, {name}!"

print(greet("Alice"))
print(greet("Bob"))
print(greet("Charlie"))
print(f"\n{greet}")

---

## 7. Context Managers: `__enter__` and `__exit__`

Context managers let you use objects with the `with` statement for automatic resource management:

In [ ]:
import time

class Timer:
    """A context manager that measures execution time."""
    
    def __init__(self, label: str = "Operation"):
        self.label = label
        self.start = None
        self.elapsed = None
    
    def __enter__(self):
        """Called at the START of 'with' block."""
        self.start = time.perf_counter()
        print(f"⏱️ [{self.label}] Starting...")
        return self  # This becomes the variable in 'with ... as var'
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Called at the END of 'with' block (even if error occurs)."""
        self.elapsed = time.perf_counter() - self.start
        print(f"⏱️ [{self.label}] Finished in {self.elapsed:.4f}s")
        return False  # Don't suppress exceptions


# Use it!
with Timer("Sum calculation") as t:
    total = sum(range(1_000_000))
    print(f"  Sum: {total:,}")

print(f"Elapsed: {t.elapsed:.4f}s")

with Timer("String building"):
    result = "-".join(str(i) for i in range(1000))

---

## 8. `__hash__` — Making Objects Usable in Sets and Dicts

If you implement `__eq__`, Python makes your object unhashable by default. You need `__hash__` too if you want to use objects in sets or as dict keys:

In [ ]:
class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y
    
    def __eq__(self, other):
        return isinstance(other, Point) and self.x == other.x and self.y == other.y
    
    def __hash__(self):
        return hash((self.x, self.y))
    
    def __repr__(self):
        return f"Point({self.x}, {self.y})"


# Now Points can be used in sets and as dict keys!
points = {Point(1, 2), Point(3, 4), Point(1, 2)}  # Duplicate removed!
print(f"Set of points: {points}")
print(f"Size: {len(points)}")  # 2, not 3

# As dict keys
labels = {
    Point(0, 0): "Origin",
    Point(1, 0): "Unit X",
    Point(0, 1): "Unit Y",
}
print(f"\nLabel at origin: {labels[Point(0, 0)]}")

---

## 9. Complete Reference: Common Dunder Methods

| Category | Method | Trigger | Description |
|----------|--------|---------|-------------|
| **Creation** | `__new__` | `MyClass()` | Creates the object |
| | `__init__` | `MyClass()` | Initializes the object |
| | `__del__` | `del obj` / GC | Destructor (cleanup) |
| **String** | `__repr__` | `repr(obj)` | Developer string |
| | `__str__` | `str(obj)`, `print()` | User string |
| | `__format__` | `f"{obj:spec}"` | Custom formatting |
| **Comparison** | `__eq__` | `==` | Equality |
| | `__lt__` | `<` | Less than |
| | `__gt__` | `>` | Greater than |
| **Arithmetic** | `__add__` | `+` | Addition |
| | `__sub__` | `-` | Subtraction |
| | `__mul__` | `*` | Multiplication |
| | `__truediv__` | `/` | Division |
| | `__neg__` | `-obj` | Unary negation |
| **Container** | `__len__` | `len(obj)` | Length |
| | `__getitem__` | `obj[key]` | Index access |
| | `__setitem__` | `obj[key] = val` | Index assignment |
| | `__contains__` | `x in obj` | Membership test |
| | `__iter__` | `for x in obj` | Iteration |
| **Other** | `__call__` | `obj()` | Make callable |
| | `__hash__` | `hash(obj)` | Hashing (sets/dicts) |
| | `__bool__` | `bool(obj)` | Truthiness |
| | `__enter__`/`__exit__` | `with obj` | Context manager |

---

## 🏋️ Practice Exercises

### Exercise 1: `Fraction` class
Create a `Fraction` class that supports:
- `__init__` with `numerator` and `denominator` (auto-simplify using GCD)
- `__add__`, `__sub__`, `__mul__`, `__truediv__`
- `__eq__`, `__lt__`
- `__repr__` and `__str__` (display as "3/4")

In [ ]:
# Exercise 1: Your code here



### Exercise 2: `Stack` class
Create a `Stack` class (LIFO) that supports:
- `push(item)` and `pop()`
- `__len__`, `__bool__`, `__contains__`
- `__iter__` (iterate from top to bottom)
- `__repr__`

In [ ]:
# Exercise 2: Your code here



---

### ✅ Solutions

In [ ]:
# Solution 1: Fraction
from math import gcd

class Fraction:
    def __init__(self, numerator: int, denominator: int):
        if denominator == 0:
            raise ValueError("Denominator cannot be zero!")
        # Simplify
        common = gcd(abs(numerator), abs(denominator))
        # Handle negative denominator
        sign = -1 if denominator < 0 else 1
        self.num = sign * numerator // common
        self.den = abs(denominator) // common
    
    def __add__(self, other):
        return Fraction(self.num * other.den + other.num * self.den,
                       self.den * other.den)
    
    def __sub__(self, other):
        return Fraction(self.num * other.den - other.num * self.den,
                       self.den * other.den)
    
    def __mul__(self, other):
        return Fraction(self.num * other.num, self.den * other.den)
    
    def __truediv__(self, other):
        return Fraction(self.num * other.den, self.den * other.num)
    
    def __eq__(self, other):
        return self.num == other.num and self.den == other.den
    
    def __lt__(self, other):
        return self.num * other.den < other.num * self.den
    
    def __repr__(self):
        return f"Fraction({self.num}, {self.den})"
    
    def __str__(self):
        if self.den == 1:
            return str(self.num)
        return f"{self.num}/{self.den}"


a = Fraction(1, 2)
b = Fraction(1, 3)

print(f"{a} + {b} = {a + b}")
print(f"{a} - {b} = {a - b}")
print(f"{a} * {b} = {a * b}")
print(f"{a} / {b} = {a / b}")
print(f"{a} == {b}: {a == b}")
print(f"{a} < {b}: {a < b}")
print(f"Fraction(4, 8) = {Fraction(4, 8)}")  # Auto-simplified to 1/2

In [ ]:
# Solution 2: Stack
class Stack:
    def __init__(self):
        self._items = []
    
    def push(self, item):
        self._items.append(item)
    
    def pop(self):
        if not self._items:
            raise IndexError("Pop from empty stack!")
        return self._items.pop()
    
    def peek(self):
        if not self._items:
            raise IndexError("Peek at empty stack!")
        return self._items[-1]
    
    def __len__(self):
        return len(self._items)
    
    def __bool__(self):
        return len(self._items) > 0
    
    def __contains__(self, item):
        return item in self._items
    
    def __iter__(self):
        return reversed(self._items)  # Top to bottom
    
    def __repr__(self):
        return f"Stack({list(reversed(self._items))})"  # Show top first

s = Stack()
s.push("A")
s.push("B")
s.push("C")

print(f"Stack: {s}")
print(f"Length: {len(s)}")
print(f"Top: {s.peek()}")
print(f"Contains 'B': {'B' in s}")
print(f"Truthiness: {bool(s)}")

print("\nPopping all:")
while s:
    print(f"  Popped: {s.pop()}")

---

## 📌 Key Takeaways

1. **Dunder methods** let objects integrate with Python's syntax and built-ins
2. **`__repr__`** for developers, **`__str__`** for users — always implement `__repr__`
3. Use **`@total_ordering`** to avoid writing all comparison methods
4. **`__getitem__`** + **`__len__`** make objects behave like containers
5. **`__call__`** makes objects callable like functions
6. **`__enter__`/`__exit__`** enable `with` statement support
7. Never call dunder methods directly — use the corresponding operators/functions

---

**⏭️ Next: [Notebook 09 — Advanced OOP Patterns](./09_Advanced_OOP_Patterns.ipynb)** — Composition, Mixins, Dataclasses, and Design Patterns!